# M3L3 E09 — Delegación paralela con LangGraph
### Módulo 3 · Lecture 3 · Sistemas Multiagente

**Ejercicio paralelo:** E04 (consulta mixta y delegación paralela)

## ¿Qué vas a aprender hoy?
- usar el LLM para detectar múltiples intenciones en una sola consulta.
- modelar fan-out dinámico con la `Send` API de LangGraph.
- usar `Annotated[list, operator.add]` para acumular resultados de múltiples ramas.


## ¿Qué necesitás saber antes?

Venís de E08 donde implementaste routing condicional (un nodo, un destino). En E09 escalamos a un nodo que dispara múltiples destinos al mismo tiempo.

> **Fan-out:** patrón donde un nodo despacha trabajo a múltiples ramas paralelas. Las ramas se ejecutan independientemente y sus resultados se fusionan en un nodo final (fan-in).

En E04 esto se hacía con un loop manual. LangGraph tiene la `Send` API para hacerlo de forma nativa:

| E04 Python puro | E09 LangGraph |
|---|---|
| Keywords para detectar dominios | LLM detecta múltiples dominios semánticamente |
| `for domain in domains: agents[domain](query)` | `[Send("specialist", {...}) for d in domains]` |
| Lista de resultados ensamblada manualmente | `Annotated[list, operator.add]` fusionado automáticamente |


## Paso 1 — Elegí tu proveedor de LLM

In [ ]:
PROVIDER = "openai"   # ← cambiá esto: "openai" | "gemini" | "claude"

import os
from getpass import getpass

if PROVIDER == "openai":
    !pip install langchain-openai -q
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

elif PROVIDER == "gemini":
    !pip install langchain-google-genai -q
    from langchain_google_genai import ChatGoogleGenerativeAI
    os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ")
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

elif PROVIDER == "claude":
    !pip install langchain-anthropic -q
    from langchain_anthropic import ChatAnthropic
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)

else:
    raise ValueError(f"PROVIDER inválido: {PROVIDER!r}. Opciones: 'openai' | 'gemini' | 'claude'")

print(f"LLM listo → proveedor: {PROVIDER}")

In [ ]:
!pip install langgraph -q

import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

print("LangGraph listo.")

## Sección 1 — La `Send` API y el State acumulador

> **Send(nodo, estado_parcial):** objeto que le indica a LangGraph que debe lanzar una rama hacia `nodo` con ese fragmento de estado. Un nodo que devuelve una lista de `Send` dispara tantas ramas como elementos tenga la lista.

El flujo del ejercicio:

```
START
  |
  v
dispatcher  -->  Send("specialist", {domain="hr",      query=...})
            -->  Send("specialist", {domain="tech",     query=...})
            -->  Send("specialist", {domain="billing",  query=...})
                       |
               (ramas paralelas)
                       |
                    merge  -->  END
```

### El problema del merge

Si tres ramas paralelas intentan escribir en el mismo campo `results`, ¿quién gana? Sin anotación, la última rama sobreescribe las otras.

```python
# Sin anotación: la última rama gana (mal)
results: list[str]

# Con anotación: cada rama agrega su valor, nunca sobreescribe (bien)
results: Annotated[list[str], operator.add]
```

In [ ]:
knowledge_base = {
    "hr": [
        "Vacaciones: cada empleado tiene 15 días hábiles por año.",
        "Licencias: registrar el pedido en PeopleOps y avisar al manager.",
    ],
    "tech": [
        "VPN: reiniciar el cliente, validar MFA y abrir ticket si persiste.",
        "Contraseña: restablecer desde el portal de identidad.",
    ],
    "billing": [
        "Facturas: cargar comprobantes antes del día 25.",
        "Reembolsos: adjuntar recibo, monto y centro de costo.",
    ],
}

def retrieve(domain: str, query: str) -> str:
    docs = knowledge_base[domain]
    words = set(query.lower().split())
    return max(docs, key=lambda d: sum(1 for w in words if w in d.lower()))

def detect_intents_llm(query: str) -> list:
    """Usa el LLM para detectar múltiples dominios en la consulta."""
    prompt = (
        "Detectá todos los dominios relevantes en esta consulta de soporte corporativo.\n"
        "Dominios válidos: hr, tech, billing\n"
        "Respondé SOLO con los dominios separados por coma, sin espacios extras.\n"
        "Si ninguno aplica, respondé exactamente: unknown\n\n"
        f"Consulta: {query}"
    )
    response = llm.invoke(prompt)
    raw = response.content.strip().lower()
    domains = [d.strip() for d in raw.split(",")]
    valid = [d for d in domains if d in ("hr", "tech", "billing")]
    return valid if valid else ["unknown"]

print("Helpers listos.")

In [ ]:
class SpecialistInput(TypedDict):
    domain: str   # qué dominio debe resolver esta rama
    query: str    # la consulta original


class ParallelState(TypedDict):
    query: str
    results: Annotated[list[str], operator.add]  # acumula: cada rama agrega, nunca sobreescribe
    final: str

## Sección 2 — Los nodos del fan-out

> **Nodo dispatcher:** devuelve una lista de `Send`, uno por dominio detectado. LangGraph interpreta esa lista y lanza las ramas en paralelo.

El nodo `specialist` recibe `SpecialistInput` (no el state completo) porque cada rama tiene su propio contexto parcial. Usa el LLM con contexto de la KB para generar una respuesta real.

**Tu TODO:** implementar `dispatcher` y `merge`.

In [ ]:
def specialist(state: SpecialistInput) -> dict:
    domain = state["domain"]
    if domain not in knowledge_base:
        return {"results": ["No puedo responder esa consulta."]}
    context = "\n".join(knowledge_base[domain])
    response = llm.invoke(
        f"Usando solo este contexto:\n{context}\n\nRespondé brevemente en español: {state['query']}"
    )
    return {"results": [f"{domain.upper()}: {response.content.strip()}"]}


def dispatcher(state: ParallelState) -> list:
    # TODO: detectar dominios con detect_intents_llm(state["query"])
    # Devolver [Send("specialist", {"domain": d, "query": state["query"]}) for d in domains]
    return []


def merge(state: ParallelState) -> dict:
    # TODO: unir state["results"] en un string con "\n" y devolver {"final": ...}
    return {"final": ""}

## Sección 3 — Construir el grafo de fan-out

La conexión `dispatcher → specialist` se declara con `add_conditional_edges`. En lugar de devolver un string, `dispatcher` devuelve una lista de `Send`. LangGraph detecta eso automáticamente y lo trata como fan-out.

```
add_conditional_edges("dispatcher", lambda x: x, ["specialist"])
                           ^              ^              ^
                        origen    fn que devuelve    nodos válidos
                                  la lista de Send
```

In [ ]:
graph = StateGraph(ParallelState)

graph.add_node("dispatcher", dispatcher)
graph.add_node("specialist",  specialist)
graph.add_node("merge",       merge)

graph.add_edge(START, "dispatcher")
graph.add_conditional_edges("dispatcher", lambda x: x, ["specialist"])
graph.add_edge("specialist", "merge")
graph.add_edge("merge", END)

app = graph.compile()
print("Grafo compilado.")

El LLM detecta múltiples dominios en la misma consulta — sin listas de keywords predefinidas.

In [ ]:
casos = [
    "tengo un problema con la VPN y necesito pedir vacaciones",
    "no puedo acceder al portal de reembolsos y perdí la contraseña",
    "qué hay para cenar",
]
for q in casos:
    r = app.invoke({"query": q, "results": [], "final": ""})
    print(f"\nQ: {q}")
    print(r["final"])

In [ ]:
print(app.get_graph().draw_mermaid())

## Checks automáticos

In [ ]:
def run_checks():
    r1 = app.invoke({"query": "tengo problema con la VPN y necesito pedir días libres", "results": [], "final": ""})
    assert "TECH" in r1["final"] and "HR" in r1["final"], f"debería tener TECH y HR: {r1['final']}"

    r2 = app.invoke({"query": "problema con la VPN", "results": [], "final": ""})
    assert "TECH" in r2["final"], f"debería tener TECH: {r2['final']}"

    print("Checks E09 OK")

run_checks()

## ¿Qué aprendiste hoy?

- El LLM detecta múltiples intenciones semánticamente: entiende "días libres" como HR sin que la palabra "vacaciones" aparezca.
- La `Send` API permite fan-out dinámico: el número de ramas se decide en runtime.
- `Annotated[list, operator.add]` acumula resultados de múltiples ramas sin sobreescribir.

## Próximo ejercicio

En **E14** vas a trasladar el support bot completo de E11 a LangGraph con LLM real.
